# Advanced Problems: Python Function Arguments

This notebook contains advanced practice problems with solutions for positional arguments, default values, `*args`, keyword-only arguments, `**kwargs`, argument forwarding, and robust function API design.

Each problem includes:

- A realistic task
- Constraints
- A solution
- Verification tests


## Problem 1 — Normalize a Flexible Record API

Write a function `make_record` that accepts:

- two required positional-or-keyword arguments: `record_id`, `name`
- any number of extra positional tags using `*tags`
- required keyword-only argument: `created_by`
- optional keyword-only argument: `active=True`
- arbitrary extra metadata using `**metadata`

Best-practice requirement: `created_by` and `active` must be keyword-only so calls stay readable.

In [1]:
def make_record(record_id, name, *tags, created_by, active=True, **metadata):
    return {
        "id": record_id,
        "name": name,
        "tags": tuple(tags),
        "created_by": created_by,
        "active": active,
        "metadata": metadata,
    }


r1 = make_record(101, "invoice", "finance", "urgent", created_by="admin", amount=250)
assert r1["tags"] == ("finance", "urgent")
assert r1["metadata"] == {"amount": 250}

r2 = make_record(record_id=102, name="profile", created_by="system", active=False)
assert r2["tags"] == ()
assert r2["active"] is False

print("Problem 1 passed")

Problem 1 passed


## Problem 2 — Diagnose Function Calls

Given the function below, determine which calls are valid and what they return. Then implement `safe_call` so it returns either the function result or the exception type/message.

```python
def analyze(a, b=10, *args, c, d=20, **kwargs):
    return a, b, args, c, d, kwargs
```

In [2]:
def analyze(a, b=10, *args, c, d=20, **kwargs):
    return a, b, args, c, d, kwargs


def safe_call(func, *args, **kwargs):
    try:
        return func(*args, **kwargs)
    except Exception as ex:
        return type(ex).__name__, str(ex)


assert safe_call(analyze, 1, c=3) == (1, 10, (), 3, 20, {})
assert safe_call(analyze, 1, 2, 3, 4, c=5, d=6, x=7) == (1, 2, (3, 4), 5, 6, {"x": 7})
assert safe_call(analyze, a=1, b=2, c=3, y=99) == (1, 2, (), 3, 20, {"y": 99})

missing_c = safe_call(analyze, 1)
assert missing_c[0] == "TypeError"
assert "required keyword-only argument" in missing_c[1]

duplicate_a = safe_call(analyze, 1, a=2, c=3)
assert duplicate_a[0] == "TypeError"
assert "multiple values" in duplicate_a[1]

print("Problem 2 passed")

Problem 2 passed


## Problem 3 — Enforce Keyword-Only Configuration

Write `connect` with this behavior:

- `host` is required
- `port` defaults to `5432`
- `user` and `password` must be keyword-only
- `timeout` is keyword-only and defaults to `30`
- extra connection options are allowed through `**options`

Security-sensitive values like `password` should not be accepted positionally.

In [3]:
def connect(host, port=5432, *, user, password, timeout=30, **options):
    return {
        "host": host,
        "port": port,
        "user": user,
        "password": password,
        "timeout": timeout,
        "options": options,
    }


cfg = connect("db.local", user="alice", password="secret")
assert cfg["port"] == 5432
assert cfg["options"] == {}

cfg = connect("db.local", 15432, user="bob", password="pw", ssl=True, retries=3)
assert cfg["port"] == 15432
assert cfg["options"] == {"ssl": True, "retries": 3}

try:
    connect("db.local", 5432, "alice", "secret")
except TypeError:
    pass
else:
    raise AssertionError("Expected TypeError when user/password are passed positionally")

print("Problem 3 passed")

Problem 3 passed


## Problem 4 — Build a Strict Wrapper Around `print`

Write `log_line(*values, sep=' ', end='\n', prefix='', suffix='', **kwargs)`.

Rules:

- It should behave like `print` for `values`, `sep`, and `end`.
- It should add `prefix` before the joined values and `suffix` after them.
- It should reject unknown keyword arguments with a helpful `TypeError`.
- It should return the final string instead of printing it.

In [4]:
def log_line(*values, sep=" ", end="\n", prefix="", suffix="", **kwargs):
    if kwargs:
        unknown = ", ".join(sorted(kwargs))
        raise TypeError(f"Unexpected keyword argument(s): {unknown}")

    body = sep.join(str(value) for value in values)
    return f"{prefix}{body}{suffix}{end}"


assert log_line(1, 2, 3) == "1 2 3\n"
assert log_line(1, 2, 3, sep="--", end="") == "1--2--3"
assert log_line("OK", prefix="[", suffix="]") == "[OK]\n"

try:
    log_line("x", flush=True)
except TypeError as ex:
    assert "flush" in str(ex)
else:
    raise AssertionError("Expected TypeError for unknown keyword argument")

print("Problem 4 passed")

Problem 4 passed


## Problem 5 — Forward Arguments Safely

Create `retry_call(func, *args, attempts=3, retry_exceptions=(Exception,), **kwargs)`.

It should call `func(*args, **kwargs)`, retry selected exceptions, return the successful result, and re-raise the last exception if all attempts fail.

Wrapper-control arguments such as `attempts` should be keyword-only.

In [5]:
def retry_call(func, *args, attempts=3, retry_exceptions=(Exception,), **kwargs):
    if attempts < 1:
        raise ValueError("attempts must be at least 1")

    last_error = None

    for _ in range(attempts):
        try:
            return func(*args, **kwargs)
        except retry_exceptions as ex:
            last_error = ex

    raise last_error


state = {"count": 0}

def flaky_add(a, b):
    state["count"] += 1
    if state["count"] < 3:
        raise RuntimeError("temporary failure")
    return a + b


assert retry_call(flaky_add, 10, 20, attempts=3) == 30
assert state["count"] == 3

try:
    retry_call(lambda: (_ for _ in ()).throw(ValueError("bad")), attempts=2)
except ValueError as ex:
    assert str(ex) == "bad"
else:
    raise AssertionError("Expected ValueError after retries are exhausted")

print("Problem 5 passed")

Problem 5 passed


## Problem 6 — Use Positional-Only and Keyword-Only Arguments

Write `format_measurement(value, unit, /, precision=2, *, label=None, signed=False)`.

Rules:

- `value` and `unit` must be positional-only.
- `precision` may be positional or keyword.
- `label` and `signed` must be keyword-only.
- Return strings like `'temperature: +23.46 C'` or `'23.46 C'`.

In [6]:
def format_measurement(value, unit, /, precision=2, *, label=None, signed=False):
    sign_flag = "+" if signed else ""
    number = f"{value:{sign_flag}.{precision}f}"
    measurement = f"{number} {unit}"
    if label is not None:
        return f"{label}: {measurement}"
    return measurement


assert format_measurement(23.456, "C") == "23.46 C"
assert format_measurement(23.456, "C", 1) == "23.5 C"
assert format_measurement(23.456, "C", precision=1) == "23.5 C"
assert format_measurement(23.456, "C", label="temperature", signed=True) == "temperature: +23.46 C"

try:
    format_measurement(value=23.456, unit="C")
except TypeError as ex:
    assert "positional-only" in str(ex)
else:
    raise AssertionError("Expected TypeError for passing positional-only parameters by keyword")

print("Problem 6 passed")

Problem 6 passed


## Problem 7 — Merge Defaults, Explicit Keywords, and Extra Metadata

Write `configure_pipeline(name, *steps, enabled=True, retries=0, **metadata)`.

Rules:

- `name` is required.
- `steps` are collected as a tuple.
- `enabled` and `retries` are keyword-only.
- `metadata` captures remaining keyword arguments.
- Reject metadata keys that collide with reserved names: `'name'`, `'steps'`, `'enabled'`, `'retries'`.

In [7]:
def configure_pipeline(name, *steps, enabled=True, retries=0, **metadata):
    reserved = {"name", "steps", "enabled", "retries"}
    collisions = reserved.intersection(metadata)

    if collisions:
        names = ", ".join(sorted(collisions))
        raise TypeError(f"Reserved metadata key(s): {names}")

    return {
        "name": name,
        "steps": tuple(steps),
        "enabled": enabled,
        "retries": retries,
        "metadata": metadata,
    }


p = configure_pipeline(
    "daily-import",
    "extract",
    "transform",
    "load",
    retries=2,
    owner="data-team",
    priority="high",
)

assert p["steps"] == ("extract", "transform", "load")
assert p["metadata"] == {"owner": "data-team", "priority": "high"}

try:
    configure_pipeline("bad", enabled=True, retries=1, steps="not allowed")
except TypeError as ex:
    assert "steps" in str(ex)
else:
    raise AssertionError("Expected TypeError for reserved metadata key")

print("Problem 7 passed")

Problem 7 passed


## Problem 8 — Implement a Mini Command Router

Write `route(command, /, *args, verbose=False, **kwargs)`.

Supported commands:

- `'add'`: returns the sum of all positional arguments plus keyword values.
- `'multiply'`: returns the product of all positional arguments plus keyword values.
- `'echo'`: returns `(args, kwargs)` unchanged.

`command` must be positional-only, and `verbose` must be keyword-only.

In [8]:
def route(command, /, *args, verbose=False, **kwargs):
    values = args + tuple(kwargs.values())

    if command == "add":
        result = sum(values)
    elif command == "multiply":
        result = 1
        for value in values:
            result *= value
    elif command == "echo":
        result = (args, kwargs)
    else:
        raise ValueError(f"Unknown command: {command!r}")

    if verbose:
        return {
            "command": command,
            "args": args,
            "kwargs": kwargs,
            "result": result,
        }

    return result


assert route("add", 1, 2, 3, x=4, y=5) == 15
assert route("multiply", 2, 3, x=4) == 24
assert route("echo", 1, 2, x=3) == ((1, 2), {"x": 3})

verbose_result = route("add", 1, 2, x=3, verbose=True)
assert verbose_result["result"] == 6
assert verbose_result["command"] == "add"

try:
    route(command="add")
except TypeError as ex:
    message = str(ex)
    assert (
        "positional-only" in message
        or "missing 1 required positional argument" in message
    )
else:
    raise AssertionError("Expected TypeError because command is positional-only")

print("Problem 8 passed")

Problem 8 passed


## Problem 9 — Preserve a Wrapped Function's Metadata

Create a decorator factory `require_keywords(*required_names)`.

It should return a decorator that wraps any function and checks that the named arguments were provided as keywords in the call.

Use `functools.wraps` so metadata such as `__name__` is preserved.

In [9]:
from functools import wraps


def require_keywords(*required_names):
    required = set(required_names)

    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            missing = required.difference(kwargs)
            if missing:
                names = ", ".join(sorted(missing))
                raise TypeError(f"These arguments must be provided as keywords: {names}")
            return func(*args, **kwargs)

        return wrapper

    return decorator


@require_keywords("user", "password")
def login(host, *, user, password):
    return host, user, password


assert login("server.local", user="alice", password="secret") == ("server.local", "alice", "secret")
assert login.__name__ == "login"

try:
    login("server.local", user="alice")
except TypeError as ex:
    assert "password" in str(ex)
else:
    raise AssertionError("Expected TypeError for missing password keyword")

print("Problem 9 passed")

Problem 9 passed


## Problem 10 — Build a Robust High/Low/Average Function

Improve the lesson's `calc_hi_lo_avg` function.

Write `calc_hi_lo_avg(*values, log_to_console=False, empty=None)`.

Rules:

- If no values are provided, return `empty`.
- Otherwise return `(min_value + max_value) / 2`.
- If `log_to_console=True`, print `high=..., low=..., avg=...`.
- `log_to_console` and `empty` must be keyword-only.

In [10]:
def calc_hi_lo_avg(*values, log_to_console=False, empty=None):
    if not values:
        return empty

    high = max(values)
    low = min(values)
    avg = (high + low) / 2

    if log_to_console:
        print(f"high={high}, low={low}, avg={avg}")

    return avg


assert calc_hi_lo_avg(1, 2, 3, 4, 5) == 3.0
assert calc_hi_lo_avg(-10, 100, 50) == 45.0
assert calc_hi_lo_avg() is None
assert calc_hi_lo_avg(empty=0) == 0
assert calc_hi_lo_avg(1, 2, 3, log_to_console=False) == 2.0

print("Problem 10 passed")

Problem 10 passed


## Final Checklist

When designing or reading a Python function signature, ask:

1. Which values are required?
2. Which values have defaults?
3. Which values should be positional-only?
4. Which values should be keyword-only?
5. Should extra positional arguments be accepted with `*args`?
6. Should extra keyword arguments be accepted with `**kwargs`?
7. Are unknown keywords safe to accept, or should they be rejected?
8. Are wrapper-specific options separated from forwarded arguments?